## Imports

In [ ]:
#package imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import seaborn as sns
import textwrap
import missingno as msno
import os
import clean
import eda
from sklearn.metrics import r2_score



## Part 1: Data Cleaning and Initial EDA


In [ ]:
#load in data and convert to dataframe

tbu_pud_csv_path = '../data/TBI PUD 10-08-2013.csv'
tbu_pud_doc_path = '../data/TBI PUD Documentation 10-08-2013.xlsx'
save_fig_path = '../figs'

if not os.path.exists(save_fig_path):
    os.makedirs(save_fig_path)

tbu_pud_raw = pd.read_csv(tbu_pud_csv_path)
tbu_pud_doc_info= pd.read_excel(tbu_pud_doc_path, skiprows = 1, nrows = 7)
tbu_pud_doc_data = pd.read_excel(tbu_pud_doc_path, skiprows = 10 )


In [ ]:
#as a sanity check, I will confirm that both dataframes contain the same variable names
print(len(list(tbu_pud_doc_data['Variable Name'])))
print(len(list(tbu_pud_raw)))
#also append the two lists and then count the unique values to make sure it matches length of both individual lists
print(len(set(list(tbu_pud_raw) + list(tbu_pud_doc_data['Variable Name']))))

In [ ]:
#Copy the data and only modify/clean the copied version
tbu_pud_cleaned = tbu_pud_raw.copy()


In [ ]:
#Observe the documentation file format that contains the variable metadata
tbu_pud_doc_data.head()

In [ ]:
#Create an accessible dataframe for the variable metadata
variable_metadata = tbu_pud_doc_data.copy()

#remove the two columns "Unnamed 4 and Unnamed 5"
variable_metadata = variable_metadata.drop(columns = ['Unnamed: 4', 'Unnamed: 5'])


#Rename column names 
variable_metadata = variable_metadata.rename(columns = {
    "Variable Name": "variable",
    "Label/Description": "description",
    "Format Name and Values": "values_key",
    "Notes": "notes"
}
)
variable_metadata = variable_metadata.set_index('variable')


In [ ]:
#Create a column for each variable to have a description name. Start with the description and work from there
variable_metadata['variable_label'] = variable_metadata['description']

#substitute acronyms to shorten label
variable_metadata['variable_label']= variable_metadata['variable_label'].str.replace('loss of consciousness', 'LOC', regex = False)
variable_metadata['variable_label'] = variable_metadata['variable_label'].str.replace('altered mental state', 'AMS', regex = False)
#remove any of the text in parenthesis
variable_metadata['variable_label'] = variable_metadata['variable_label'].str.replace(r'\s*\([^)]*\)', '', regex=True)
variable_metadata['values_key'] = variable_metadata['values_key'].str.replace(r'\s*\([^)]*\)', '', regex=True)

#some variables still have very long labels, so abbreviate those manually
variable_metadata.loc['Amnesia_verb','variable_label'] = 'Amnesia?'
variable_metadata.loc['Observed','variable_label'] = 'Patient observed in ED after initial eval?'
variable_metadata.loc['Drugs', 'variable_label'] = 'Suspicion of drugs/alcohol?'
variable_metadata.loc['ActNorm', 'variable_label'] = 'Is the child acting normally?'
variable_metadata.loc['Clav', 'variable_label'] = 'Trauma above the clavicles'
variable_metadata.loc['OSI', 'variable_label'] = 'Other substantial injuries'
# variable_metadata.loc[]


#Add line breaks to the labels with more than 25 characters
variable_metadata['variable_label'] = variable_metadata['variable_label'].apply(
    lambda x: textwrap.fill(x, width=25) if isinstance(x, str) else x
)



In [ ]:

#Add a column to variable metadata that just shows values in the values_key
def extract_ints(val):
    return [int(num) for num in re.findall(r'\d+', str(val))]

variable_metadata['values_extracted'] = variable_metadata['values_key'].apply(extract_ints)

#Find statistical values of each column. Find the number of missing values for each variable
variable_stats = tbu_pud_cleaned.describe().transpose()
variable_metadata = variable_metadata.merge(variable_stats, left_on= 'variable', right_index = True)

#add a column to metadata for number of missing values
missing_vals = tbu_pud_cleaned.isnull().sum()
variable_metadata['missing_vals'] = missing_vals


In [ ]:
#Based on the domain knowledge, there are specific variables that were used as predictors for ciTBI
#generate a list of variable predictors specific to ages below and above 2. 
variable_predictors_below2 = ['AMS','Hema', 'High_impact_InjSev', 'LOCSeparate', 'SFxPalp', 'ActNorm' ]
variable_predictors_above2 = ['AMS', 'High_impact_InjSev', 'LOCSeparate', 'Vomit', 'SFxBas', 'HASeverity']

#Generate a list of all the variable predictors in the study
variable_predictors = list(set(variable_predictors_above2+ variable_predictors_below2))


In [ ]:
#Impute missing data based on information from the domain. See report for detailed explanation of assumptions.
tbu_pud_cleaned = clean.impute_domain_knowledge(tbu_pud_cleaned)

In [ ]:
#View the general value counts of the predictor variables for the raw dataset

eda.plot_value_counts(df = tbu_pud_raw, variable_metadata=variable_metadata, columns = variable_predictors)

In [ ]:
%load_ext autoreload
%autoreload 2
fig,ax = eda.plot_missing_vals(tbu_pud_raw, plot_title= 'Missing Values in Raw Data', max_vars = 15, variable_metadata = variable_metadata)
fig.savefig(os.path.join(save_fig_path, 'missing_vals_raw'), dpi = 300, bbox_inches = 'tight')


In [ ]:
#visualize as a heatmap
# Calculate missing value percentages
missing_data = tbu_pud_raw.isnull().mean()

# See the columns with greater than 1% missing data to see where the issues are
cols_to_plot = missing_data[(missing_data > 0.01) & (missing_data < 0.99)].index

# Plot only the filtered columns
ax = msno.heatmap(tbu_pud_raw[cols_to_plot], cmap = 'coolwarm')
ax.figure.savefig(os.path.join(save_fig_path, 'missing_vals_headmap.png'), dpi = 300, bbox_inches = 'tight')

#correlation closer to 1 indicates that two variables tend to be missing together. From the heatmap, its clear that GCS Verbal, GCSEye, and GCSMotor are often missing together. Other variables have a correlation between -.1 and .3, so there is not much correlation.

In [ ]:
#After imputing missing data based on assumptions from domain knowledge, we can observe the remaining missing data from our variable predictors:
eda.plot_missing_vals(tbu_pud_raw[variable_predictors], tbu_pud_cleaned[variable_predictors], labels = ['Raw Data', 'Cleaned Data'], figsize = (4,6), plot_title= 'Missing Data for Variable Predictors', xlim = 5000)

### Focus list for data cleaning
Below is a list of variables I paid most attention to for data cleaning 

Variable Predictors for children younger than 2: 
- Altered mental status
  - GCS = Glasgow Coma Scale
    - GCSEye
    - GCSVerbal
    - GCSMotor
    - GCSTotal
    - GCSGroup
  - AMS
    - AMS
    - AMSAgitated
    - AMSSleep
    - AMSSlow
    - AMSRepeat
    - AMSOth
- Scalp haematoma
  - Hema
  - HemaLoc
  - HemaSize
- Loss of Consciousness
  - LOCSeparate
  - LocLen
- Mechanism of Injury
  - High_impact_InjSev
- Palpable or unclear skull fracture
  - SFxPalp
  - SFxPalpDepress
- Acting normally per parent
  - ActNorm

**Predictors for children 2 and up:**

- Altered mental status
- Loss of Consciousness
  - LOCSeparate
  - LocLen
- History of Vomiting
  - Vomit
- Mechanism of Injury
  - High_impact_InjSev
- Basilar skull fracture
  - SFxBas
  - SFxBasHem
  - SFxBasOto
  - SFxBasPer
  - SFxBasRet
  - SFxBasRhi
- Severe headache
  - HASeverity

In [ ]:
#Now export the cleaned data to a figure
%load_ext autoreload
%autoreload 2

fig,ax = eda.plot_value_counts(df = tbu_pud_raw, df2 = tbu_pud_cleaned, labels = ['Raw Data', 'Cleaned data'] ,variable_metadata=variable_metadata, columns =['High_impact_InjSev', 'HASeverity',  'ActNorm'], fontsize = 13)
fig.savefig(os.path.join(save_fig_path, 'imputed_data_comparison.png'), dpi = 300, bbox_inches = 'tight')

In [ ]:
#Right now, 'ActNorm' = 1 suggests child is acting normal and =0 suggests not acting normally. I want to flip the data so that =1 is not acting normally, and = 0 is normal.
#I'm doing this because the other yes/no observations all have =1 as "yes", indicating a predictor is present.
#For ActNorm, "No" is actually the indication that the predictor is present, so I want to change it to "yes" so that its more similar to the other predictors.
tbu_pud_cleaned['ActNorm'] = tbu_pud_cleaned['ActNorm'].map({0: 1, 1: 0})

#update variable_key as well

variable_metadata.loc['ActNorm', 'values_key'] = "0 Yes\n1 No"

In [ ]:
#There are still quite a bit of data missing. For remaining missing data, there are a few options I considered:

#Option 1, drop the remaining null values from the dataset:
clean_data_drop_na = clean.drop_missing_rows(tbu_pud_cleaned, columns = tbu_pud_cleaned.columns)

#Option 2, replace the null values with the median from the observed data:
clean_data_med = clean.impute_from_median(tbu_pud_cleaned, variable_metadata= variable_metadata, columns = tbu_pud_cleaned.columns)

#Option 3, replace null values with a randomly sampled value from the observed data:
clean_data_sampled = clean.impute_from_distribution(tbu_pud_cleaned, columns = tbu_pud_cleaned.columns)

## Exploratory Data Analysis

In [ ]:
#First I will check the variable_predictors in the cleaned datasets against the raw data
%load_ext autoreload
%autoreload 2
# eda.plot_value_counts(df = tbu_pud_raw, df2 = clean_data_drop_na, labels = ['Raw Data', 'Cleaned data with \n Null Values Removed'] ,variable_metadata=variable_metadata, columns = variable_predictors[0:6])
# eda.plot_value_counts(df = tbu_pud_raw, df2 = clean_data_med, labels = ['Raw Data', 'Cleaned data with \n Null Values Substituted\n With Median'] ,variable_metadata=variable_metadata, columns = variable_predictors[0:6])
fig, ax = eda.plot_value_counts(df = tbu_pud_raw, df2 = clean_data_sampled, labels = ['Raw Data', 'Cleaned data with \n Null Values Sampled \n from Observations'] ,variable_metadata=variable_metadata, columns = variable_predictors[0:6], fontsize = 11)
fig.savefig(os.path.join(save_fig_path, 'imputed_data_comparison.png'))


In [ ]:
clean_data_sampled.head()

In [ ]:
#hand calc to check function for calculating percentages
num_predictor = (len(tbu_pud_raw[tbu_pud_raw['AMS'] == 1]))
num_predictor_with_target = (len(tbu_pud_raw[(tbu_pud_raw['AMS'] == 1) & (tbu_pud_raw['PosIntFinal'] == 1)]))
# print(len(tbu_pud_raw[(tbu_pud_raw['PosIntFinal'] == 1)]))

num_predictor_with_target/num_predictor


In [ ]:
# expand the variable predictors list to consider other potentially influential variables
variable_predictors_expanded = variable_predictors.copy()  # or list(variable_predictors)
variable_predictors_expanded += ['Clav', 'NeuroD', 'OSI', 'Seiz', 'Amnesia_verb']

In [ ]:

#Plot the correlations between the predictors and if there was a CiTBI

#Calculate the correlations for each variable 
%load_ext autoreload
%autoreload 2

# List the variables that should use the severity threshold (3) instead of yes/no (1)
severity_vars = ['HASeverity', 'High_impact_InjSev'] 

patients_with_ciTBI = eda.get_pct_summary_given_predictor(clean_data_sampled, predictors = variable_predictors_expanded, severity_vars = severity_vars)

#Generate a summary table of patients who had each observation coincident with having a ciTBI
summary_df = pd.DataFrame.from_dict(patients_with_ciTBI, orient='index', columns=['Percent with ciTBI'])
summary_df.index.name = 'Variable'
label_map = variable_metadata['variable_label'].to_dict()
summary_df['Description'] = [label_map.get(var, var) for var in summary_df.index]

summary_df = summary_df.sort_values('Percent with ciTBI', ascending=False)
summary_df['Percent with ciTBI'] = summary_df['Percent with ciTBI'].round(2)

# Reorder columns so Description comes first
summary_df = summary_df[['Description', 'Percent with ciTBI']]

In [ ]:
#understand why a CT was ordered compared to what the CT finding actually was
%load_ext autoreload
%autoreload 2

#adjust labels for plot
variable_metadata.loc['ActNorm', 'variable_label'] = 'Child is not acting normally'
variable_metadata['variable_label'] = variable_metadata['variable_label'].str.replace('?', '', regex=False)
# Replace newlines with spaces first, so word-matching works
variable_metadata['variable_label'] = variable_metadata['variable_label'].str.replace('\n', ' ', regex=False)
variable_metadata['variable_label'] = variable_metadata['variable_label'].str.replace(
    'If a CT is being ordered or obtained check the most important indications in influencing the decision to obtain a head CT: ',
    '', regex=False
)
variable_metadata.loc['IndRqstTrauma','variable_label'] = 'Trauma Team Request'
variable_metadata.loc['IndRqstMD','variable_label'] = 'MD Request'
#make labels title case
variable_metadata['variable_label'] = variable_metadata['variable_label'].str.title()

#generate the plot
fig, axes= eda.plot_ct_yield_by_indication(clean_data_sampled, variable_metadata)
fig.savefig(os.path.join(save_fig_path, 'ct_findings_by_age'))
# print(no_finding_df)

In [ ]:
%load_ext autoreload
%autoreload 2
#make a box plot for variables
fig,ax = eda.plot_box_whiskers(clean_data_sampled, variable_to_plot ='AgeinYears', variable_predictors = variable_predictors[0:3], variable_metadata = variable_metadata, fontsize = 16)

fig.savefig(os.path.join(save_fig_path, 'box_whiskers_age.png'))

In [ ]:
# Explore how many patients had CT scans but no actual findings
patients_with_ct = clean_data_sampled[clean_data_sampled['CTDone'] == 1]

# Amount of patients with all non-applicable findings
finding_cols = [f'Finding{i}' for r in [range(1, 15), range(20, 24)] for i in r]

# Identify patients with "0" for all findings
no_findings_mask = (patients_with_ct[finding_cols] == 0).all(axis=1)

# Subset of patients with no findings at all
patients_no_findings = patients_with_ct[no_findings_mask]

# Count
n_no_findings = no_findings_mask.sum()
pct_no_findings = n_no_findings / len(patients_with_ct) * 100

print(f"Patients with no findings: {n_no_findings} out of {len(patients_with_ct)} ({pct_no_findings}%)")

In [ ]:
%load_ext autoreload
%autoreload 2
#Plot the most common findings
fig, ax = eda.plot_findings(clean_data_sampled, variable_metadata, patient_filter_col='CTDone', plot_title = 'Most Common CT Findings Among Patients with Head CT', fontsize = 10)
fig.savefig(os.path.join(save_fig_path, 'ct_findings.png'))

In [ ]:
%load_ext autoreload
%autoreload 2
#Plot the most common reasons CT scans were ordered
fig, ax = eda.plot_findings(clean_data_sampled, variable_metadata, patient_filter_col='CTForm1', plot_title = 'Most Common Reasons for Ordering CT', top_n = 8, fontsize = 13)
fig.savefig(os.path.join(save_fig_path, 'ct_reason_for_order.png'))

In [ ]:
#Calculate the correlations for each variable with whether or not patient had a ciTBI
%load_ext autoreload
%autoreload 2
fig, ax = eda.variable_tbi_heatmap(clean_data_sampled,plot_title = 'Percent of Patients with Observation given ciTBI',  variable_metadata=variable_metadata, variables_to_plot=variable_predictors_expanded, split_by_age=True, vmax = 100, fontsize = 10)
fig.savefig(os.path.join(save_fig_path, 'variable_tbi_heatmap'))


# Stability Check

In [ ]:
#adjust label names for plotting
variable_metadata.loc['AMS','variable_label'] = 'GCS <15 or Other\n Signs of AMS'
variable_metadata.loc['ActNorm', 'variable_label'] = 'Child is not \n acting normally'
variable_metadata.loc['Clav', 'variable_label'] = 'Trauma above the clavicles'
variable_metadata.loc['Hema', 'variable_label'] = 'Scalp Hematoma/Swelling'

In [ ]:
#present a perturbed version of the findings


#Compare the heatmap with cleaned_data_med
#here, null values are replaced with median values
fig, ax = eda.variable_tbi_heatmap(clean_data_med,plot_title = '',  variable_metadata=variable_metadata, variables_to_plot=variable_predictors_expanded, split_by_age=True, vmax = 100)
fig.savefig(os.path.join(save_fig_path, 'heatmap_perturbation_med.png'), dpi = 1000, bbox_inches = 'tight')

In [ ]:
#Calculate the correlations for each variable 
%load_ext autoreload
%autoreload 2
fig, ax = eda.variable_tbi_heatmap(clean_data_sampled,plot_title = '',  variable_metadata=variable_metadata, variables_to_plot=variable_predictors_expanded, split_by_age=True, vmax = 100)
fig.savefig(os.path.join(save_fig_path, 'heatmap_base'), dpi = 1000, bbox_inches = 'tight')

In [ ]:
#Compare the heatmap with cleaned_data_med
#here, null values are removed
fig, ax = eda.variable_tbi_heatmap(clean_data_drop_na,plot_title = 'Null Values Removed',  variable_metadata=variable_metadata, variables_to_plot=variable_predictors_expanded, split_by_age=True, vmax = 100)
fig.savefig(os.path.join(save_fig_path, 'heatmap_perturbation_null_removed.png'))

## Modeling

In [ ]:
# Applying the PECARN Model

age_group = clean_data_sampled['AgeTwoPlus']
# group 1 is younger than 2, group 2 is 2 and up

# Children under 2
# if patient has hematoma or not acting normally then they are recommended CT scan
under2_yes1 = np.logical_or(
    clean_data_sampled['AMS'] == 1,
    clean_data_sampled['SFxPalp'] == 1
)

# if those conditions aren't true, they still might be recommended CT scan for other reasons
under2_yes2 = pd.concat([
    clean_data_sampled['Hema'] == 1,
    clean_data_sampled['LOCSeparate'] == 1,
    clean_data_sampled['High_impact_InjSev'] == 3,
    clean_data_sampled['ActNorm'] == 1  # note earlier i changed ActNorm = 1 to mean child is not acting normally
], axis=1).any(axis=1)


over2_yes1 = np.logical_or(
    clean_data_sampled['AMS'] == 1,
    clean_data_sampled['SFxBas'] == 1
)

over2_yes2 = pd.concat([
    clean_data_sampled['LOCSeparate'] == 1,
    clean_data_sampled['Vomit'] == 1,
    clean_data_sampled['HASeverity'] == 3,
    clean_data_sampled['High_impact_InjSev'] == 3
], axis=1).any(axis=1)

# depending on age group, assign whether or not they should have a CT or might be eligible
recommend_ct = np.where(age_group == 1, under2_yes1, over2_yes1)
consider_ct = np.where(age_group == 1, under2_yes2, over2_yes2)

# determine if patient should have ct
clean_data_sampled['ct_recommend_pecarn'] = np.select(
    condlist=[age_group.isna(), recommend_ct, consider_ct],
    choicelist=['no', 'yes', 'yes, with other considerations'],
    default='no'
)


print(clean_data_sampled['ct_recommend_pecarn'].value_counts())

In [ ]:
#now try CHALICE model

#set the criteria
chalice_criteria = pd.concat([
    clean_data_sampled['AMS'] == 1, #AMS
    clean_data_sampled['LocLen'] == 4, #greater than 5 min
    clean_data_sampled['Amnesia_verb'] == 1, #amnesia observed
    clean_data_sampled['VomitNbr'] == 3, #more than 2 vomit episodes
    clean_data_sampled['Seiz'] == 1, #seizure
    clean_data_sampled['SFxPalp'] == 1,# palpable skull fracture
    clean_data_sampled['SFxBas'] == 1,  # signs of basilar skull fracture
    clean_data_sampled['NeuroD'] == 1, # focal neurological deficit
    clean_data_sampled['High_impact_InjSev'] == 3, #severe injury
], axis=1).any(axis=1)

# Determine if patient meets CHALICE criteria for CT
clean_data_sampled['ct_recommend_chalice'] = np.select(
    condlist=[chalice_criteria],
    choicelist=['yes'],
    default='no'
)

print(clean_data_sampled['ct_recommend_chalice'].value_counts())

In [ ]:
#plot comparison of the results 
# Flags for whether each model recommends CT
pecarn_flag = clean_data_sampled['ct_recommend_pecarn'].isin(['yes', 'maybe'])
chalice_flag = clean_data_sampled['ct_recommend_chalice'] == 'yes'
citbi = clean_data_sampled['PosIntFinal']

plot_df = pd.DataFrame({
    'ciTBI Status': ['Yes', 'Yes', 'No', 'No'],
    'Model': ['PECARN', 'CHALICE', 'PECARN', 'CHALICE'],
    'Percent': [
        pecarn_flag[citbi == 1].mean() * 100,
        chalice_flag[citbi == 1].mean() * 100,
        pecarn_flag[citbi == 0].mean() * 100,
        chalice_flag[citbi == 0].mean() * 100,
    ]
})


fig, ax = plt.subplots(figsize=(8, 6))
ax.grid(axis='y', color='lightgray', linewidth=0.5, zorder=0)

sns.barplot(
    data=plot_df, x='ciTBI Status', y='Percent', hue='Model',
    palette=['cornflowerblue', 'forestgreen'], ax=ax, zorder=3
)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize = 16)

ax.set_ylabel('Percent of Patients Recommended for CT', fontsize = 16)
ax.set_xlabel('Presence of ciTBI', fontsize = 16)
ax.set_ylim(0, 110)
ax.set_title('Comparison of PECARN and CHALICE', fontsize=16, fontweight='bold')
ax.tick_params(direction='in', labelsize=14)
ax.legend(title='', fontsize=14)

plt.tight_layout()
plt.show()

fig.savefig(os.path.join(save_fig_path, 'chalice_vs_pecarn.png'))

In [ ]:
#check again with data perturbation, null values replaced with median.

# Applying the PECARN Model

age_group = clean_data_med['AgeTwoPlus']
# group 1 is younger than 2, group 2 is 2 and up

# Children under 2
# if patient has hematoma or not acting normally then they are recommended CT scan
under2_yes1 = np.logical_or(
    clean_data_med['AMS'] == 1,
    clean_data_med['SFxPalp'] == 1
)

# if those conditions aren't true, they still might be recommended CT scan for other reasons
under2_yes2 = pd.concat([
    clean_data_med['Hema'] == 1,
    clean_data_med['LOCSeparate'] == 1,
    clean_data_med['High_impact_InjSev'] == 3,
    clean_data_med['ActNorm'] == 1  # note earlier i changed ActNorm = 1 to mean child is not acting normally
], axis=1).any(axis=1)


over2_yes1 = np.logical_or(
    clean_data_med['AMS'] == 1,
    clean_data_med['SFxBas'] == 1
)

over2_yes2 = pd.concat([
    clean_data_med['LOCSeparate'] == 1,
    clean_data_med['Vomit'] == 1,
    clean_data_med['HASeverity'] == 3,
    clean_data_med['High_impact_InjSev'] == 3
], axis=1).any(axis=1)

# depending on age group, assign whether or not they should have a CT or might be eligible
recommend_ct = np.where(age_group == 1, under2_yes1, over2_yes1)
consider_ct = np.where(age_group == 1, under2_yes2, over2_yes2)

# determine if patient should have ct
clean_data_med['ct_recommend_pecarn'] = np.select(
    condlist=[age_group.isna(), recommend_ct, consider_ct],
    choicelist=['no', 'yes', 'yes, with other considerations'],
    default='no'
)

#now try CHALICE model

#set the criteria
chalice_criteria = pd.concat([
    clean_data_med['AMS'] == 1, #AMS
    clean_data_med['LocLen'] == 4, #greater than 5 min
    clean_data_med['Amnesia_verb'] == 1, #amnesia observed
    clean_data_med['VomitNbr'] == 3, #more than 2 vomit episodes
    clean_data_med['Seiz'] == 1, #seizure
    clean_data_med['SFxPalp'] == 1,# palpable skull fracture
    clean_data_med['SFxBas'] == 1,  # signs of basilar skull fracture
    clean_data_med['NeuroD'] == 1, # focal neurological deficit
    clean_data_med['High_impact_InjSev'] == 3, #severe injury
], axis=1).any(axis=1)

# Determine if patient meets CHALICE criteria for CT
clean_data_med['ct_recommend_chalice'] = np.select(
    condlist=[chalice_criteria],
    choicelist=['yes'],
    default='no'
)

#plot comparison of the results 
# Flags for whether each model recommends CT
pecarn_flag = clean_data_med['ct_recommend_pecarn'].isin(['yes', 'maybe'])
chalice_flag = clean_data_med['ct_recommend_chalice'] == 'yes'
citbi = clean_data_med['PosIntFinal']

plot_df = pd.DataFrame({
    'ciTBI Status': ['Yes', 'Yes', 'No', 'No'],
    'Model': ['PECARN', 'CHALICE', 'PECARN', 'CHALICE'],
    'Percent': [
        pecarn_flag[citbi == 1].mean() * 100,
        chalice_flag[citbi == 1].mean() * 100,
        pecarn_flag[citbi == 0].mean() * 100,
        chalice_flag[citbi == 0].mean() * 100,
    ]
})


fig, ax = plt.subplots(figsize=(8, 6))
ax.grid(axis='y', color='lightgray', linewidth=0.5, zorder=0)

sns.barplot(
    data=plot_df, x='ciTBI Status', y='Percent', hue='Model',
    palette=['cornflowerblue', 'forestgreen'], ax=ax, zorder=3
)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize = 16)

ax.set_ylabel('Percent of Patients Recommended for CT', fontsize = 16)
ax.set_xlabel('Presence of ciTBI', fontsize = 16)
ax.set_ylim(0, 110)
ax.set_title('Comparison of PECARN and CHALICE (perturbation)', fontsize=16, fontweight='bold')
ax.tick_params(direction='in', labelsize=14)
ax.legend(title='', fontsize=14)

plt.tight_layout()
plt.show()

fig.savefig(os.path.join(save_fig_path, 'chalice_vs_pecarn_perturbation.png'))
